# MusicScope™ — Executive Notebook (24 Charts + Momentum)
## THIS IS A PROPOSED NOTEBOOK FORMAT!!! TO BE USED FOR CONSIDERATION IN BUILDING A NOTEBOOK

**How we read:** punchline-first titles → direct labels → color for signal, grey for context.  
**For recruiters/A&Rs:** automatic “DEMO DATA” banner/watermark until live sources are detected.

In [ ]:
from __future__ import annotations
import os, re, math, warnings, time, calendar
from dataclasses import dataclass
from typing import Optional, List, Tuple, Dict

import numpy as np
import pandas as pd

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.dates import DateFormatter, AutoDateLocator
from matplotlib.colors import ListedColormap
from IPython.display import HTML, display

# Optional interactivity for the bar race
try:
    import plotly.express as px
    PLOTLY_AVAILABLE = True
except Exception:
    PLOTLY_AVAILABLE = False
    warnings.warn("Plotly not installed. Install with: pip install plotly")

# ---------- Styling: clean, accessible, exec-ready
mpl.rcParams.update({
    "figure.dpi": 140,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "legend.frameon": False,
})
GREY_3 = "#8E8E8E"; GREY_2 = "#B0B0B0"; GREY_1 = "#D7D7D7"
POS_C  = "#1B9E77"; NEG_C  = "#D95F02"; ACCENT = "#E7298A"; BLUE_HI = "#1f77b4"
PALETTE = ["#1B9E77","#7570B3","#D95F02","#E7298A","#66A61E","#E6AB02","#A6761D","#666666"]
DATE_FMT = DateFormatter("%b %d %Y")  # always show year

def pick_color(i:int) -> str:
    return PALETTE[i % len(PALETTE)]

def slide(figsize=(11,6), watermark: Optional[str]=None):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_anchor("NW")
    if watermark:
        fig.text(0.5, 0.5, watermark, color=GREY_1, fontsize=60, ha="center",
                 va="center", alpha=0.35, rotation=30)
    return fig, ax

def action_title(ax: mpl.axes.Axes, finding: str, implication: str, action: str) -> None:
    ax.set_title(f"{finding} → {implication} → {action}")

def direct_line_labels(ax: mpl.axes.Axes, fontsize: int = 10):
    lines = [ln for ln in ax.get_lines() if not ln.get_label().startswith("_")]
    for ln in lines:
        x, y = ln.get_xdata(), ln.get_ydata()
        if len(x)==0: continue
        ax.annotate(ln.get_label(), xy=(x[-1], y[-1]), xytext=(5,0), textcoords="offset points",
                    va="center", fontsize=fontsize, color=ln.get_color(), fontweight="bold")
    if ax.get_legend(): ax.get_legend().remove()

def label_bars(ax: mpl.axes.Axes, fmt="{:.0f}", fontsize=10):
    for p in ax.patches:
        h = p.get_height()
        if h == 0: continue
        ax.text(p.get_x()+p.get_width()/2, p.get_y()+h, fmt.format(h),
                ha="center", va="bottom", fontsize=fontsize, color="#222")

def iso8601_to_seconds(iso: str) -> int:
    if not isinstance(iso, str): return 0
    h = m = s = 0
    mobj = re.match(r"PT(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", iso)
    if mobj:
        h = int(mobj.group(1) or 0); m = int(mobj.group(2) or 0); s = int(mobj.group(3) or 0)
    return h*3600 + m*60 + s

In [ ]:
# Expecting:
# vids columns: video_id, title, published_at (datetime), view_count, like_count, comment_count, duration_iso
# comments columns: video_id, published_at (datetime), text
IS_DEMO_DATA = False

def make_demo(seed=7):
    rng = np.random.default_rng(seed)
    dates = pd.date_range("2024-03-01", periods=60, freq="4D")
    dv = pd.DataFrame({
        "video_id": [f"vid_{i:03d}" for i in range(len(dates))],
        "title": [f"Demo Video {i+1}" for i in range(len(dates))],
        "published_at": dates,
        "view_count": rng.integers(10_000, 400_000, len(dates)).astype(int),
        "like_count": rng.integers(300, 12_000, len(dates)).astype(int),
        "comment_count": rng.integers(40, 1500, len(dates)).astype(int),
        "duration_iso": ["PT" + str(rng.integers(120, 420)) + "S" for _ in range(len(dates))],
    })
    rows = []
    for _, r in dv.iterrows():
        n = int(np.clip(r["comment_count"], 20, 400))
        ts = r["published_at"] + pd.to_timedelta(rng.integers(0, max(3, (pd.Timestamp.today()-r["published_at"]).days)), unit="D")
        for i in range(n):
            dt = ts + pd.to_timedelta(int(rng.integers(0, 7)), unit="D")
            txt = rng.choice(["this is fire", "mid but catchy", "production goes crazy",
                              "didn’t vibe with the hook", "lyrics are deep", "visuals slap",
                              "meh", "banger", "could be mixed better"])
            rows.append({"video_id": r["video_id"], "published_at": dt, "text": txt})
    dc = pd.DataFrame(rows)
    return dv, dc

if 'vids' not in globals() or 'comments' not in globals():
    vids, comments = make_demo()
    IS_DEMO_DATA = True

# Coerce dtypes & safe features
vids = vids.copy()
vids["published_at"] = pd.to_datetime(vids["published_at"], errors="coerce")
vids = vids.dropna(subset=["published_at"]).sort_values("published_at").reset_index(drop=True)
vids["duration_sec"] = vids["duration_iso"].map(iso8601_to_seconds)
END_DATE = pd.Timestamp.today().normalize()
vids["age_days"] = (END_DATE - vids["published_at"]).dt.days.clip(lower=1)
vids["views_per_day"] = (vids["view_count"] / vids["age_days"]).replace([np.inf, np.nan], 0.0)
vids["like_rate"] = (vids["like_count"] / vids["view_count"].replace(0, np.nan)).fillna(0.0).clip(0,1)
vids["publish_week"] = vids["published_at"].dt.to_period("W").dt.start_time
vids["publish_month"] = vids["published_at"].dt.to_period("M").dt.start_time

comments = comments.copy()
comments["published_at"] = pd.to_datetime(comments["published_at"], errors="coerce")
comments = comments.dropna(subset=["published_at"]).sort_values("published_at").reset_index(drop=True)
comments["_date"] = comments["published_at"].dt.floor("D")

display(HTML(
    '<div style="padding:16px;border-radius:12px;'
    + ('border:3px solid #d95f02;background:#fff3e6;font-weight:700;">⚠️ DEMO DATA ACTIVE — replace with live YouTube v3 pulls.'
       if IS_DEMO_DATA else
       'border:2px solid #1b9e77;background:#eef9f5;">✅ Live YouTube v3 data detected.')
    + '</div>'
))

In [ ]:
# Momentum components (v3-accessible): views/day, like_rate, recent comment velocity
c_daily = (comments.groupby(["video_id","_date"]).size()
           .rename("comments_d").reset_index())

# Build a daily panel per video
rows = []
for _, g in vids.iterrows():
    d_range = pd.date_range(g["published_at"].normalize(), END_DATE, freq="D")
    base = pd.DataFrame({"video_id": g["video_id"], "date": d_range})
    base = base.merge(c_daily[c_daily["video_id"]==g["video_id"]][["video_id","_date","comments_d"]],
                      left_on=["video_id","date"], right_on=["video_id","_date"], how="left").drop(columns=["_date"])
    base["comments_d"] = base["comments_d"].fillna(0).astype(int)
    base["views_per_day"] = g["views_per_day"]
    base["like_rate"]     = g["like_rate"]
    base["title"]         = g["title"]
    rows.append(base)
momentum_daily = pd.concat(rows, ignore_index=True)

# Rolling comment velocity + robust day-wise normalization → 0–100
momentum_daily["cmt_14d"] = (momentum_daily.groupby("video_id")["comments_d"]
                             .transform(lambda s: s.rolling(14, min_periods=3).sum()))
momentum_daily["comments_per_day_14d"] = (momentum_daily["cmt_14d"]/14.0).fillna(0)

def _score_component_daily(df, col):
    med = df[col].median()
    mad = (df[col] - med).abs().median() + 1e-9
    z = (df[col] - med) / (1.4826*mad)
    return (z.rank(pct=True)*100).clip(0,100)

momentum_daily = momentum_daily.groupby("date", group_keys=False).apply(
    lambda d: d.assign(
        s_views=_score_component_daily(d, "views_per_day"),
        s_like=_score_component_daily(d, "like_rate"),
        s_cmtv=_score_component_daily(d, "comments_per_day_14d"),
    )
)
momentum_daily["momentum_score"] = (0.45*momentum_daily["s_views"]
                                    +0.25*momentum_daily["s_like"]
                                    +0.30*momentum_daily["s_cmtv"]).round(1)
momentum_daily["state"] = np.where(momentum_daily["momentum_score"]>=55, "pre_breakout", "baseline")

# Weekly sentiment shares for Chart 1 (neg/neu/pos from quick lexicon)
def _quick_sent(text: str) -> str:
    t = (text or "").lower()
    pos_kw = ["fire","banger","crazy","deep","slap","love","great","amazing","dope"]
    neg_kw = ["meh","mid","bad","trash","worse","hate"]
    if any(k in t for k in pos_kw): return "pos"
    if any(k in t for k in neg_kw): return "neg"
    return "neu"

if "sentiment" not in comments.columns:
    comments["sentiment"] = comments["text"].map(_quick_sent)

sent_week = (comments.assign(week=comments["_date"].dt.to_period("W").dt.start_time)
             .groupby("week")["sentiment"].value_counts(normalize=True)
             .rename("share").reset_index()
             .pivot(index="week", columns="sentiment", values="share")
             .reindex(columns=["neg","neu","pos"]).fillna(0.0).reset_index())

# Episode detection (true breakout ≥60) + pre-breakout warning hours (55–60 before start)
def breakout_episodes(df: pd.DataFrame, pre_th=55.0, br_th=60.0) -> pd.DataFrame:
    d = df[["video_id","date","momentum_score"]].dropna().sort_values(["video_id","date"]).copy()
    d["_ge55"] = (d["momentum_score"] >= pre_th).astype(int)
    d["_ge60"] = (d["momentum_score"] >= br_th).astype(int)

    grp_artist_change = (d["video_id"] != d["video_id"].shift()).cumsum()
    grp_state_change  = (d["_ge60"] != d["_ge60"].shift()).cumsum()
    d["_run_id"] = grp_artist_change + grp_state_change

    br = d[d["_ge60"]==1].copy()
    if br.empty:
        return pd.DataFrame(columns=["video_id","start","end","days","pre_warning_hours"])

    eps = (br.groupby(["video_id","_run_id"])
             .agg(start=("date","min"), end=("date","max"))
             .reset_index(level="_run_id", drop=True).reset_index())
    eps["days"] = (eps["end"] - eps["start"]).dt.days + 1

    warn_hours = []
    for _, r in eps.iterrows():
        sub = d[(d["video_id"]==r["video_id"]) & (d["date"] <= r["start"])]
        if sub.empty: warn_hours.append(0); continue
        sub["state_pre"] = ((sub["momentum_score"] >= pre_th) & (sub["momentum_score"] < br_th)).astype(int)
        cur = r["start"] - pd.Timedelta(days=1); streak = 0
        idx = sub.set_index("date")["state_pre"]
        while cur in idx.index and int(idx.loc[cur])==1:
            streak += 1; cur -= pd.Timedelta(days=1)
        warn_hours.append(streak*24)
    eps["pre_warning_hours"] = warn_hours
    return eps

episodes = breakout_episodes(momentum_daily, pre_th=55, br_th=60) \
             .merge(vids[["video_id","title"]], on="video_id", how="left") \
             .sort_values(["pre_warning_hours","days"], ascending=False).reset_index(drop=True)

# DEMO banner text for figures
WM = "DEMO DATA" if IS_DEMO_DATA else None

In [ ]:
fig, ax = slide(figsize=(11,7), watermark=WM)
df = sent_week.copy()
y = np.arange(len(df))
ax.barh(y, -df["neg"]*100, color=NEG_C, label="Negative")
ax.barh(y,  df["pos"]*100, color=POS_C, label="Positive")
ax.barh(y,  df["neu"]*100, left=-df["neu"]*50, color=GREY_2, alpha=0.35, height=0.85)  # neutral de-emphasized

ax.set_yticks(y[::2], [d.strftime("%b %d %Y") for d in df["week"][::2]])
ax.axvline(0, color="#333", lw=1)
ax.set_xlabel("Share of comments (%) — negative ← 0 → positive")
ax.set_xlim(-100, 100)
action_title(ax, "Positives lead; neutral mass managed",
                  "neutral can mask polarity swings",
                  "act on sustained pos/neg streaks, not one-off blips")
plt.show()

In [ ]:
wk = vids.groupby("publish_week").size().rename("uploads")
fig, ax = slide(watermark=WM)
ax.bar(wk.index, wk.values, color=GREY_2, width=6)
label_bars(ax, fmt="{:.0f}")
ax.xaxis.set_major_formatter(DATE_FMT)
action_title(ax, "Upload cadence steady by week",
                 "consistency feeds discovery",
                 "lock next 6 weeks on Tue/Thu cadence")
plt.show()

In [ ]:
ts = (vids.set_index("published_at")["views_per_day"].resample("W").mean())
fig, ax = slide(watermark=WM)
ax.plot(ts.index, ts.values, lw=2.3, color=pick_color(0), label="Avg views/day (by publish week)")
ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
action_title(ax, "Recent publishes earn higher views/day",
                 "format learning curve",
                 "standardize on winning structure")
plt.show()

In [ ]:
lr = (vids.set_index("published_at")["like_rate"].resample("W").median()*100)
fig, ax = slide(watermark=WM)
ax.plot(lr.index, lr.values, lw=2.3, color=pick_color(1), label="Weekly median like rate (%)")
ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
action_title(ax, "Like rate inching up",
                 "social proof compounds",
                 "pin hearts, invite reactions")
plt.show()

In [ ]:
cv = comments.set_index("_date").resample("W").size().rename("comments")
fig, ax = slide(watermark=WM)
ax.plot(cv.index, cv.values, lw=2.2, color=pick_color(2), label="Weekly comments")
ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
action_title(ax, "Comment volume rising",
                 "community energy increasing",
                 "host Q&A in comments on peak weeks")
plt.show()

In [ ]:
cv = comments.set_index("_date").resample("W").size().rename("comments")
fig, ax = slide(watermark=WM)
ax.plot(cv.index, cv.values, lw=2.2, color=pick_color(2), label="Weekly comments")
ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
action_title(ax, "Comment volume rising",
                 "community energy increasing",
                 "host Q&A in comments on peak weeks")
plt.show()

In [ ]:
lengths = comments["text"].str.len().clip(0, 1000)
fig, ax = slide(watermark=WM)
ax.hist(lengths, bins=30, color=GREY_2, edgecolor="#333")
ax.set_xlabel("Comment length (characters)"); ax.set_ylabel("Count")
action_title(ax, "Short comments dominate",
                 "snackable prompts perform",
                 "ask 1-line questions to boost replies")
plt.show()

In [ ]:
stop = set("the a an and to for of in on is it this that with my your our i you we they are be was were as at from by".split())
toks = (comments["text"].str.lower().str.replace(r"[^a-z0-9\s]", " ", regex=True).str.split())
words = pd.Series([w for lst in toks.dropna() for w in lst if w not in stop and len(w)>=3])
top = words.value_counts().head(20)[::-1]
fig, ax = slide(figsize=(11,7), watermark=WM)
ax.barh(top.index, top.values, color=GREY_2)
action_title(ax, "Conversation leans craft & vibe",
                 "themes reveal content hooks",
                 "seed prompts around top terms")
plt.show()

In [ ]:
bad = {"http","https","subscribe","follow","free","giveaway"}
flagged = comments.assign(flag=comments["text"].str.lower().apply(lambda t: int(any(b in t for b in bad))))
wk = flagged.groupby(pd.Grouper(key="_date", freq="W"))["flag"].mean()*100
fig, ax = slide(watermark=WM)
ax.plot(wk.index, wk.values, lw=2.2, color=NEG_C, label="% flagged words")
ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
action_title(ax, "Spam-leaning phrases stable",
                 "moderation predictable",
                 "pre-filter bait terms")
plt.show()

In [ ]:
def _lang_guess(t):
    t = (t or "").lower()
    if any(w in t for w in ["el","la","que","con","para","pero"]): return "es"
    if any(w in t for w in ["le","la","et","pour","mais"]): return "fr"
    return "en"
if "lang" not in comments.columns:
    comments["lang"] = comments["text"].map(_lang_guess)

lang = comments["lang"].value_counts(normalize=True).head(8)*100
fig, ax = slide(watermark=WM)
ax.bar(lang.index, lang.values, color=[pick_color(i) for i in range(len(lang))])
label_bars(ax, fmt="{:.0f}%")
action_title(ax, "Language mix clarifies comms plan",
                 "speak the audience’s language",
                 "subtitle & post in top langs")
plt.show()

In [ ]:
loc = vids.copy(); loc["hour"] = loc["published_at"].dt.hour
med = loc.groupby("hour")["views_per_day"].median()
fig, ax = slide(watermark=WM)
bars = ax.bar(med.index, med.values, color=GREY_2)
peak = med.idxmax(); bars[peak].set_color(ACCENT)
action_title(ax, f"Publishes around {peak}:00 perform best",
                 "align drops to online peaks",
                 "test ±1h around peak")
plt.show()

In [ ]:
loc = vids.copy(); loc["dow"] = loc["published_at"].dt.day_name()
order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
med = loc.groupby("dow")["views_per_day"].median().reindex(order)
fig, ax = slide(watermark=WM)
bars = ax.bar(med.index, med.values, color=[GREY_2]*7)
bars[med.values.argmax()].set_color(ACCENT)
action_title(ax, f"{med.index[med.values.argmax()]} leads median views/day",
                 "schedule for lift",
                 "standardize publish day")
plt.xticks(rotation=20); plt.show()

In [ ]:
eng = pd.DataFrame({"video": vids["title"], "likes": vids["like_count"], "comments": vids["comment_count"]})
eng = eng.melt("video", var_name="type", value_name="count")
fig, ax = slide(figsize=(11,7), watermark=WM)
for i,(k,grp) in enumerate(eng.groupby("type")):
    ax.bar(grp.index + i*0.35, grp["count"], width=0.35, label=k, color=pick_color(i))
ax.set_xticks(range(len(vids))); ax.set_xticklabels(vids["title"], rotation=90)
action_title(ax, "Engagement skews to likes",
                 "low-friction signals dominate",
                 "pin comments to invite replies")
plt.show()

In [ ]:
cc = comments["author"].fillna("anon").value_counts().head(15)[::-1] if "author" in comments.columns else \
     comments.groupby(comments.index//999).size().head(15)[::-1]
fig, ax = slide(figsize=(11,7), watermark=WM)
ax.barh(cc.index, cc.values, color=GREY_2)
action_title(ax, "Top commenters = loyalty nodes",
                 "community flywheel",
                 "reward repeat commenters")
plt.show()

In [ ]:
rep = (comments.groupby("author").size() > 1).rename("is_returning") if "author" in comments.columns else \
      pd.Series(dtype=bool)
join = comments.merge(rep.reset_index(), on="author", how="left") if "author" in comments.columns else \
       comments.assign(is_returning=False)
share = join.groupby(pd.Grouper(key="_date", freq="W"))["is_returning"].mean().fillna(0)*100
fig, ax = slide(watermark=WM)
ax.plot(share.index, share.values, lw=2.2, color=pick_color(4), label="% returning commenters")
ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
action_title(ax, "Returning commenters increasing",
                 "relationship depth rising",
                 "feature fan comments in Shorts")
plt.show()

In [ ]:
lens = vids["title"].str.len()
fig, ax = slide(watermark=WM)
ax.scatter(lens, vids["views_per_day"], s=26, alpha=0.45, color=pick_color(5))
ax.set_xlabel("Title length (chars)"); ax.set_ylabel("Views per day")
action_title(ax, "Moderate title lengths perform best",
                 "clarity > cleverness",
                 "test trims around median length")
plt.show()

In [ ]:
counts = vids["title"].str.count(r"#")
fig, ax = slide(watermark=WM)
ax.scatter(counts, vids["views_per_day"], s=26, alpha=0.45, color=pick_color(6))
ax.set_xlabel("Hashtag count in title"); ax.set_ylabel("Views per day")
action_title(ax, "Hashtags show limited effect",
                 "content wins",
                 "optimize hook/thumbnail vs tags")
plt.show()

In [ ]:
vpm = (vids["view_count"] / (vids["duration_sec"].replace(0, np.nan)/60)).replace([np.inf, np.nan], 0)
fig, ax = slide(watermark=WM)
ax.scatter(vids["duration_sec"]/60, vpm, s=26, alpha=0.45, color=pick_color(0))
ax.set_xlabel("Duration (minutes)"); ax.set_ylabel("Views per minute (lifetime)")
action_title(ax, "2–6 min formats most efficient",
                 "tight edits convert",
                 "hit the hook early; trim intros")
plt.show()

In [ ]:
if not PLOTLY_AVAILABLE:
    raise RuntimeError("Plotly not installed. pip install plotly")

race = momentum_daily.copy()
race["date_str"] = race["date"].dt.strftime("%b %d %Y")
race["invest"] = np.where(race["momentum_score"]>=55, "Invest", "")

fig = px.bar(
    race.sort_values(["date","momentum_score"]),
    x="momentum_score", y="title",
    orientation="h",
    color="state",
    text="invest",
    animation_frame="date_str",
    title="Momentum Race — BLUE = Pre-Breakout (≥55) • invest labels",
    range_x=[0, max(60, float(race["momentum_score"].max())+5)],
    color_discrete_map={"pre_breakout": BLUE_HI, "baseline": GREY_2},
    height=720
)
fig.update_traces(textposition="outside")
fig.update_layout(legend_title_text="", yaxis={"categoryorder":"total ascending"})
fig.show()

In [ ]:
# Join latest momentum per video
latest = momentum_daily.sort_values("date").groupby("video_id").tail(1)
join = vids.merge(latest[["video_id","momentum_score","state"]], on="video_id", how="left")
fig, ax = slide(watermark=WM)
ax.scatter(join["views_per_day"], join["momentum_score"], s=28, alpha=0.55,
           color=np.where(join["state"]=="pre_breakout", BLUE_HI, GREY_2))
ax.set_xlabel("Views per day"); ax.set_ylabel("Momentum score (0–100)")
action_title(ax, "Momentum aligns with views/day but rewards discussion quality",
                 "balanced signal",
                 "seed prompts for comments in the 50–60 band")
plt.show()

In [ ]:
vv = vids.sort_values("published_at").copy()
vv["gap_days"] = vv["published_at"].diff().dt.days.fillna(0)
fig, ax = slide(watermark=WM)
ax.scatter(vv["gap_days"], vv["views_per_day"], s=26, alpha=0.45, color=pick_color(3))
ax.set_xlabel("Gap to previous publish (days)"); ax.set_ylabel("Views per day")
action_title(ax, "Shorter gaps correlate with higher views/day",
                 "cadence matters",
                 "avoid >14d gaps unless strategic")
plt.show()

In [ ]:
md = momentum_daily.copy()
share = (md.assign(is_pre=(md["momentum_score"]>=55).astype(int))
           .groupby("date")["is_pre"].mean()*100)
fig, ax = slide(watermark=WM)
ax.plot(share.index, share.values, lw=2.3, color=BLUE_HI, label="% of videos ≥55 per day")
ax.xaxis.set_major_formatter(DATE_FMT); direct_line_labels(ax)
action_title(ax, "More videos hitting pre-breakout",
                 "portfolio momentum rising",
                 "budget small tests for 55–60 zone")
plt.show()

In [ ]:
# Top recent episodes (last 30d)
recent_eps = episodes[episodes["end"] >= (END_DATE - pd.Timedelta(days=30))].head(12)
md2 = momentum_daily.assign(is_br=(momentum_daily["momentum_score"]>=60).astype(int))
last30 = pd.date_range(END_DATE - pd.Timedelta(days=30), END_DATE, freq="D")
hot = (md2[md2["date"].isin(last30)].groupby("date")["is_br"].sum()
       .sort_values(ascending=False).head(4))

fig, (ax1, ax2) = plt.subplots(2,1, figsize=(11,9))

# Panel 1 — Breakout duration
ax1.barh(recent_eps["title"], recent_eps["days"], color=NEG_C)
for y, d in zip(recent_eps["title"], recent_eps["days"]):
    ax1.text(d, y, f" {int(d)}d", va="center", ha="left", fontsize=10, color="#222")
ax1.set_xlabel("Days in breakout (≥60)")
ax1.set_title("Breakout duration spiking → shift budget now", fontweight="bold")

# Panel 2 — Pre-breakout warning hours
top_warn = episodes.sort_values("pre_warning_hours", ascending=False).head(12)
ax2.barh(top_warn["title"], top_warn["pre_warning_hours"], color=POS_C)
for y, h in zip(top_warn["title"], top_warn["pre_warning_hours"]):
    ax2.text(h, y, f" {int(h)}h", va="center", ha="left", fontsize=10, color="#222")
ax2.set_xlabel("Pre-breakout warning time (hours where 55≤score<60)")
ax2.set_title("Warning windows lengthening → monitor & seed before the jump", fontweight="bold")

fig.suptitle("KPI 22 — Breakout duration & pre-breakout warning time", fontweight="bold")
fig.text(0.02, 0.01, "Hot days: " + ", ".join([d.strftime("%b %d %Y") for d in hot.index]),
         color=NEG_C, fontsize=10)
plt.tight_layout(); plt.show()

In [ ]:
daily_breakouts = md2.groupby("date")["is_br"].sum()
last = daily_breakouts.index.max()
first_of_month = (last - pd.offsets.MonthBegin(1)).normalize()
prev_month_end = (first_of_month - pd.Timedelta(days=1))
month_start = (prev_month_end - pd.offsets.MonthBegin(1)).normalize() + pd.offsets.MonthBegin(0)
month_end = month_start + pd.offsets.MonthEnd(0)

rng = pd.date_range(month_start, month_end, freq="D")
vals = daily_breakouts.reindex(rng).fillna(0).astype(int)

wks = calendar.Calendar().monthdayscalendar(month_start.year, month_start.month)
mat = np.zeros((len(wks), 7), dtype=int)
for i,week in enumerate(wks):
    for j,day in enumerate(week):
        if day != 0:
            d = pd.Timestamp(year=month_start.year, month=month_start.month, day=day)
            mat[i,j] = int(vals.get(d, 0))

fig, ax = slide(figsize=(11,5), watermark=WM)
cmap = ListedColormap(["#D7D7D7","#E6AB02","#D95F02","#E7298A"])  # grey → gold/orange/magenta
im = ax.imshow(mat, cmap=cmap, aspect="auto")
ax.set_xticks(range(7)); ax.set_xticklabels(["Mon","Tue","Wed","Thu","Fri","Sat","Sun"])
ax.set_yticks(range(len(wks))); ax.set_yticklabels([f"Wk {i+1}" for i in range(len(wks))])
ax.set_title(f"Daily breakout intensity — {month_start.strftime('%b %Y')} → cluster promo on hot days")
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        if mat[i,j]>0:
            ax.text(j, i, str(mat[i,j]), ha="center", va="center", fontsize=10, color="#222")
plt.colorbar(im, ax=ax, fraction=0.02, pad=0.02, label="# of videos ≥60")
plt.show()

In [ ]:
fig, axes = plt.subplots(2,2, figsize=(11,8))
series = [
    ("Views/day (weekly)", vids.set_index("published_at")["views_per_day"].resample("W").mean(), pick_color(0)),
    ("Like rate % (weekly median)", vids.set_index("published_at")["like_rate"].resample("W").median()*100, pick_color(1)),
    ("Comments/14d/day (latest)", momentum_daily.groupby("video_id")["comments_per_day_14d"].tail(1), pick_color(2)),
    ("Momentum (latest)", momentum_daily.groupby("video_id")["momentum_score"].tail(1), pick_color(3)),
]
for ax,(title, s, color) in zip(axes.ravel(), series):
    if isinstance(s.index, pd.DatetimeIndex):
        ax.plot(s.index, s.values, lw=2.0, color=color, label=title)
        ax.xaxis.set_major_formatter(DATE_FMT)
    else:
        ax.scatter(range(len(s)), s.values, s=26, alpha=0.55, color=color, label=title)
    direct_line_labels(ax); ax.set_title(title)
fig.suptitle("KPIs — trend clarity at a glance → act on outliers", fontweight="bold")
fig.tight_layout(); plt.show()